# data_upload_&_ml_model



## Startup cells

In [0]:
# Set environment variables for sagemaker_studio imports

import os
os.environ['DataZoneProjectId'] = 'd8zhnxy6csojyi'
os.environ['DataZoneDomainId'] = 'dzd-46a78g55i6dgzu'
os.environ['DataZoneEnvironmentId'] = '5t8tfb0js4s362'
os.environ['DataZoneDomainRegion'] = 'ca-central-1'

# create both a function and variable for metadata access
_resource_metadata = None

def _get_resource_metadata():
    global _resource_metadata
    if _resource_metadata is None:
        _resource_metadata = {
            "AdditionalMetadata": {
                "DataZoneProjectId": "d8zhnxy6csojyi",
                "DataZoneDomainId": "dzd-46a78g55i6dgzu",
                "DataZoneEnvironmentId": "5t8tfb0js4s362",
                "DataZoneDomainRegion": "ca-central-1",
            }
        }
    return _resource_metadata
metadata = _get_resource_metadata()

In [0]:
"""
Logging Configuration

Purpose:
--------
This sets up the logging framework for code executed in the user namespace.
"""

from typing import Optional


def _set_logging(log_dir: str, log_file: str, log_name: Optional[str] = None):
    import os
    import logging
    from logging.handlers import RotatingFileHandler

    level = logging.INFO
    max_bytes = 5 * 1024 * 1024
    backup_count = 5

    # fallback to /tmp dir on access, helpful for local dev setup
    try:
        os.makedirs(log_dir, exist_ok=True)
    except Exception:
        log_dir = "/tmp/kernels/"

    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, log_file)

    logger = logging.getLogger() if not log_name else logging.getLogger(log_name)
    logger.handlers = []
    logger.setLevel(level)

    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")

    # Rotating file handler
    fh = RotatingFileHandler(filename=log_path, maxBytes=max_bytes, backupCount=backup_count, encoding="utf-8")
    fh.setFormatter(formatter)
    logger.addHandler(fh)

    logger.info(f"Logging initialized for {log_name}.")


_set_logging("/var/log/computeEnvironments/kernel/", "kernel.log")
_set_logging("/var/log/studio/data-notebook-kernel-server/", "metrics.log", "metrics")

In [0]:
import logging
from sagemaker_studio import ClientConfig, sqlutils, sparkutils, dataframeutils

logger = logging.getLogger(__name__)
logger.info("Initializing sparkutils")
spark = sparkutils.init()
logger.info("Finished initializing sparkutils")

In [0]:
def _reset_os_path():
    """
    Reset the process's working directory to handle mount timing issues.
    
    This function resolves a race condition where the Python process starts
    before the filesystem mount is complete, causing the process to reference
    old mount paths and inodes. By explicitly changing to the mounted directory
    (/home/sagemaker-user), we ensure the process uses the correct, up-to-date
    mount point.
    
    The function logs stat information (device ID and inode) before and after
    the directory change to verify that the working directory is properly
    updated to reference the new mount.
    
    Note:
        This is executed at module import time to ensure the fix is applied
        as early as possible in the kernel initialization process.
    """
    try:
        import os
        import logging

        logger = logging.getLogger(__name__)
        logger.info("---------Before------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)

        os.chdir("/home/sagemaker-user")

        logger.info("---------After------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)
    except Exception as e:
        logger.exception(f"Failed to reset working directory: {e}")

_reset_os_path()

## Notebook

In [0]:
# Upload data to InventoryTransactions table
import boto3
import pandas as pd
from decimal import Decimal

dynamodb = boto3.resource('dynamodb')
table = dynamodb.Table('InventoryTransactions')

df = pd.read_csv("online_retail_transactions.csv")

with table.batch_writer() as batch:
    for _, row in df.iterrows():
        item = {
            'StockCode': str(row['StockCode']),
            'TransactionKey': str(row['TransactionKey']),
            'WarehouseID': str(row['WarehouseID']),
            'Invoice': str(row['Invoice']),
            'CustomerID': str(row['Customer ID']),
            'InvoiceDate': str(row['InvoiceDateStr']),
            'Quantity': int(row['Quantity']),
            'Price': Decimal(str(row['Price'])),
            'Country': str(row['Country']) if pd.notna(row['Country']) else '',
            'Description': str(row['Description']) if pd.notna(row['Description']) else ''
        }
        batch.put_item(Item=item)

print("Upload complete")

Upload complete


In [0]:
# Check total number of items in InventoryTransactions table
import boto3

dynamodb = boto3.resource('dynamodb')
table = dynamodb.Table('InventoryTransactions')

count = 0
response = table.scan(Select='COUNT')
count += response['Count']

while 'LastEvaluatedKey' in response:
    response = table.scan(
        Select='COUNT',
        ExclusiveStartKey=response['LastEvaluatedKey']
    )
    count += response['Count']

print("Total items in DynamoDB:", count)

Total items in DynamoDB: 407695


In [0]:
# Upload inventory_products.csv to DynamoDB
import boto3
import pandas as pd
from decimal import Decimal

dynamodb = boto3.resource('dynamodb')
table = dynamodb.Table('InventoryProducts')

df = pd.read_csv("inventory_products.csv")

with table.batch_writer() as batch:
    for _, row in df.iterrows():
        item = {
            'StockCode': str(row['StockCode']),
            'WarehouseID': str(row['WarehouseID']),
            'CurrentStock': int(row['CurrentStock']),
            'AveragePrice': Decimal(str(row['AveragePrice'])),
            'LastUpdated': str(row['LastUpdated']),
            'Description': str(row['Description']) if pd.notna(row['Description']) else '',
            'Country': str(row['Country']) if pd.notna(row['Country']) else ''
        }
        batch.put_item(Item=item)

print("Inventory summary upload complete")

Inventory summary upload complete


In [0]:
# Verify the total number of rows in InventoryProducts table
import boto3

dynamodb = boto3.resource('dynamodb')
table = dynamodb.Table('InventoryProducts')

count = 0

response = table.scan(Select='COUNT')
count += response['Count']

while 'LastEvaluatedKey' in response:
    response = table.scan(
        Select='COUNT',
        ExclusiveStartKey=response['LastEvaluatedKey']
    )
    count += response['Count']

print("Total rows in DynamoDB:", count)

Total rows in DynamoDB: 7416


In [0]:
#Creates the feature engineering notebook/script
import pandas as pd

df = pd.read_csv("daily_sales.csv")

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["StockCode"] = df["StockCode"].astype(str)

# Sort properly
df = df.sort_values(["StockCode", "InvoiceDate"])

# Time features
df["DayOfWeek"] = df["InvoiceDate"].dt.dayofweek
df["Month"] = df["InvoiceDate"].dt.month
df["Year"] = df["InvoiceDate"].dt.year

# Lag features
df["Lag1"] = df.groupby("StockCode")["Quantity"].shift(1)
df["Lag7"] = df.groupby("StockCode")["Quantity"].shift(7)

# Rolling mean
df["Rolling7Mean"] = (
    df.groupby("StockCode")["Quantity"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

# Drop rows with missing lag features
df = df.dropna()

print(df.head())
print(df.shape)

/tmp/ipykernel_57/4159611231.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("daily_sales.csv")


   StockCode         InvoiceDate  Quantity  ...  Lag1  Lag7  Rolling7Mean
7      10002 2009-12-04 17:31:00         1  ...  48.0  12.0     13.000000
8      10002 2009-12-06 11:57:00        48  ...   1.0   2.0     11.428571
9      10002 2009-12-06 15:24:00         1  ...  48.0   1.0     18.000000
10     10002 2009-12-07 16:40:00         2  ...   1.0   4.0     18.000000
11     10002 2009-12-08 15:14:00        12  ...   2.0  12.0     17.714286

[5 rows x 9 columns]
(368878, 9)


In [0]:
# StockCode encoding.
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
df["StockCodeEncoded"] = encoder.fit_transform(df["StockCode"])

In [0]:
# Split features and target
X = df[[
    "StockCodeEncoded",
    "DayOfWeek",
    "Month",
    "Year",
    "Lag1",
    "Lag7",
    "Rolling7Mean"
]]

y = df["Quantity"]

In [0]:
# Train/test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [0]:
# Installing XGBoost model
pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/131.7 MB ? eta -:--:--

   ━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/131.7 MB 54.6 MB/s eta 0:00:03

   ━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/131.7 MB 79.4 MB/s eta 0:00:02

   ━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/131.7 MB 85.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━ 74.4/131.7 MB 96.4 MB/s eta 0:00:01

...

   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 1/2 [xgboost]

   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 1/2 [xgboost]

   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 1/2 [xgboost]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]


Note: you may need to restart the kernel to use updated packages.


In [0]:
# Train the XGBoost model
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [0]:
# Evaluate the model
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 13.240317344665527
RMSE: 90.97023680509467


In [0]:
# Save the model and encoder

import joblib

joblib.dump(model, "demand_forecast_model.pkl")
joblib.dump(encoder, "stockcode_encoder.pkl")

['stockcode_encoder.pkl']

In [0]:
# Reorder calculation helper
def calculate_reorder(current_stock, predicted_demand):
    safety_stock = max(5, round(predicted_demand * 0.2))
    recommended_reorder_qty = max(0, predicted_demand + safety_stock - current_stock)
    low_stock_alert = current_stock < (predicted_demand + safety_stock)

    return {
        "predictedDemand": predicted_demand,
        "safetyStock": safety_stock,
        "recommendedReorderQty": recommended_reorder_qty,
        "lowStockAlert": low_stock_alert
    }

In [0]:
# Test with an example
result = calculate_reorder(current_stock=20, predicted_demand=70)
print(result)

{'predictedDemand': 70, 'safetyStock': 14, 'recommendedReorderQty': 64, 'lowStockAlert': True}


In [0]:
# Uploading the .pkl files to S3 bucket
import boto3

s3 = boto3.client("s3")

bucket_name = "smart-inventory-cloud-project"

s3.upload_file("demand_forecast_model.pkl", bucket_name, "models/demand_forecast_model.pkl")
s3.upload_file("stockcode_encoder.pkl", bucket_name, "models/stockcode_encoder.pkl")

print("Model files uploaded to S3 successfully.")

Model files uploaded to S3 successfully.


In [0]:
import boto3
from boto3.dynamodb.conditions import Key
import pandas as pd

# Connect to DynamoDB
dynamodb = boto3.resource('dynamodb')
table = dynamodb.Table('InventoryProducts')

# Query for StockCode = '22107'
items = []

response = table.query(
    KeyConditionExpression=Key('StockCode').eq('22107')
)

items.extend(response.get('Items', []))

# Handle pagination (if more data exists)
while 'LastEvaluatedKey' in response:
    response = table.query(
        KeyConditionExpression=Key('StockCode').eq('22107'),
        ExclusiveStartKey=response['LastEvaluatedKey']
    )
    items.extend(response.get('Items', []))

# Print results
print(f"Total items found: {len(items)}")
for item in items:
    print(item)

# Convert to DataFrame (optional)
df = pd.DataFrame(items)

print("\nDataFrame Preview:")
display(df.head())

Total items found: 2
{'Country': 'EIRE', 'LastUpdated': '2010-11-19T13:54:00', 'AveragePrice': Decimal('3.65'), 'Description': 'PIZZA PLATE IN BOX', 'StockCode': '22107', 'CurrentStock': Decimal('56'), 'WarehouseID': 'W1'}
{'Country': 'United Kingdom', 'LastUpdated': '2010-12-09T13:08:00', 'AveragePrice': Decimal('3.73'), 'Description': 'PIZZA PLATE IN BOX', 'StockCode': '22107', 'CurrentStock': Decimal('516'), 'WarehouseID': 'W3'}

DataFrame Preview:


,Country,LastUpdated,AveragePrice,Description,StockCode,CurrentStock,WarehouseID
0,EIRE,2010-11-19T13:54:00,3.65,PIZZA PLATE IN BOX,22107,56,W1
1,United Kingdom,2010-12-09T13:08:00,3.73,PIZZA PLATE IN BOX,22107,516,W3


## Shutdown cells

In [0]:
"""
Stop spark session and associated Athena Spark session
"""

from IPython import get_ipython as _get_ipython
_get_ipython().user_ns["spark"].stop()